In [ ]:
import cv2
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import os
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:

model1 = load_model('best_model_binary.h5')
model2 = load_model('best_model_VGG.h5')
model3 = load_model('//content/sample_data/best_model.h5')  # Multi-class model


OSError: Unable to synchronously open file (truncated file: eof = 109051904, sblock->base_addr = 0, stored_eof = 616828512)

In [ ]:
# --- Image Preprocessing ---
def preprocess_image_model1(img_path):
    img = image.load_img(img_path, target_size=(400, 400), color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.
    return img_array

def preprocess_image_model2(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    if len(img.shape) == 2 or img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.
    return img_array

def preprocess_image_model3(img_path):
    img = cv2.imread(img_path)  # Or use PIL if needed, but cv2 is faster for resizing
    img = cv2.resize(img, (224, 224))  # Adjust target size as needed for model3
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.
    return img_array


# --- Classification Function ---
def classify_image(img_path, binary_class_names=['all', 'noall'], multi_class_names=['defect1', 'defect2', 'no_defect']):
    """Classifies an image using all three models and combines the results."""

    # --- Binary Classification (Model 1 & 2) ---
    img_array1 = preprocess_image_model1(img_path)
    prediction1 = model1.predict(img_array1)
    class_index1 = np.argmax(prediction1)
    confidence1 = prediction1[0][class_index1]

    img_array2 = preprocess_image_model2(img_path)
    prediction2 = model2.predict(img_array2)
    class_index2 = np.argmax(prediction2)
    confidence2 = prediction2[0][class_index2]

    # Combine binary results (majority voting)
    if class_index1 == class_index2:
        binary_result = (binary_class_names[class_index1], max(confidence1, confidence2), "Confident")
    else:
        if confidence1 > confidence2:
            binary_result = (binary_class_names[class_index1], confidence1, "Model 1")
        else:
            binary_result = (binary_class_names[class_index2], confidence2, "Model 2")

    # --- Multi-class Classification (Model 3) ---
    img_array3 = preprocess_image_model3(img_path)
    prediction3 = model3.predict(img_array3)
    class_index3 = np.argmax(prediction3)
    confidence3 = prediction3[0][class_index3]
    multi_class_result = (multi_class_names[class_index3], confidence3)

    return binary_result, multi_class_result

# --- Directory Processing ---
def process_directory(dir_path, binary_class_names=['all', 'noall'], multi_class_names=['defect1', 'defect2', 'no_defect']):
    """Processes a directory of images and generates statistics and reports."""

    image_files = [f for f in os.listdir(dir_path) if f.endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))]
    num_images = len(image_files)

    binary_class_counts = {name: 0 for name in binary_class_names}
    binary_confident_counts = {name: 0 for name in binary_class_names}
    multi_class_counts = {name: 0 for name in multi_class_names}  # Counts for each defect type

    details = []  # List to store detailed results (filename, binary result, multi-class result)


    for image_file in image_files:
        image_path = os.path.join(dir_path, image_file)
        (binary_result, multi_class_result) = classify_image(image_path, binary_class_names, multi_class_names)

        # Update counts
        binary_class_name = binary_result[0]
        binary_class_counts[binary_class_name] += 1
        if binary_result[2] == "Confident":
            binary_confident_counts[binary_class_name] += 1

        multi_class_name = multi_class_result[0]
        multi_class_counts[multi_class_name] += 1

        details.append((image_file, binary_result, multi_class_result))

    # --- Generate Statistics ---
    print("\n--- Summary Statistics ---")
    print(f"Total images processed: {num_images}")
    print("\n--- Binary Classification ---")
    for name in binary_class_names:
        print(f"  {name}: {binary_class_counts[name]} (Confident: {binary_confident_counts[name]})")
    print("\n--- Multi-class Classification (Defect Types) ---")
    for name in multi_class_names:
        print(f"  {name}: {multi_class_counts[name]}")


    # --- Generate Histogram (Defect Types) ---
    plt.figure(figsize=(10, 6))  # Adjust figure size as needed
    plt.bar(multi_class_counts.keys(), multi_class_counts.values())
    plt.xlabel("Defect Type")
    plt.ylabel("Number of Images")
    plt.title("Distribution of Defect Types")
    plt.xticks(rotation=45, ha="right")  # Rotate x-axis labels for readability
    plt.tight_layout()  # Adjust layout to prevent labels from overlapping
    plt.savefig("defect_histogram.png")  # Save the histogram to a file
    plt.show()


    # --- Generate Defect Table (Pandas DataFrame) ---
    defect_data = []
    for filename, binary_result, multi_class_result in details:
        defect_data.append({
            "Filename": filename,
            "Binary Class": binary_result[0],
            "Binary Confidence": binary_result[1],
            "Binary Decision": binary_result[2],
            "Defect Type": multi_class_result[0],
            "Defect Confidence": multi_class_result[1]
        })
    df = pd.DataFrame(defect_data)
    print("\n--- Defect Details Table ---")
    print(df.to_string())  # Print the entire DataFrame
    df.to_csv("defect_details.csv", index=False) # Save to CSV file

    return df



In [ ]:

# --- Main Execution ---
if __name__ == "__main__":
    dir_path = input("Enter the directory containing the images: ") # Get directory from user
    if not os.path.isdir(dir_path):
        print("Invalid directory path.")
    else:
        # Set your class names here (important!)
        binary_class_names = ['all', 'noall']
        multi_class_names = ['attrition', 'label', 'nodefect', 'pores', 'scratch'] # Example names!  REPLACE THEM!

        df = process_directory(dir_path, binary_class_names, multi_class_names)
        print("Processing complete.  Check 'defect_histogram.png' and 'defect_details.csv'.")

In [ ]:
   if __name__ == "__main__":
       dir_path = input("Enter the directory containing the images: ") # Get directory from user
       if not os.path.isdir(dir_path):
           print("Invalid directory path.")
       else:
           # Set your class names here (important!)
           binary_class_names = ['all', 'noall']  # Replace with your binary class names
           multi_class_names = ['defect1', 'defect2', 'no_defect', 'other_defect'] # Example
           df = process_directory(dir_path, binary_class_names, multi_class_names)
           print("Processing complete.  Check 'defect_histogram.png' and 'defect_details.csv'.")